In [ ]:
!pip install nltk
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

In [3]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import re

stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

def preprocess_text(text):

    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)

    words = word_tokenize(text)
    processed_words = [ps.stem(word) for word in words if word not in stop_words]
    return " ".join(processed_words)


In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained Sentence-BERT model
# 'all-MiniLM-L6-v2' is a good balance of speed and performance
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating Sentence-BERT embeddings for documents...")

In [ ]:
!pip install gradio

In [ ]:
!pip install sentence-transformers

In [ ]:
import gradio as gr
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import re
import nltk
import os
from sentence_transformers import SentenceTransformer

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

stop_words = set(stopwords.words('english'))
ps = PorterStemmer()
sbert_model = SentenceTransformer('all-MiniLM-L6-v2') # Load model once globally

def preprocess_text_for_ui(text):
    text = text.lower()
    text = re.sub(r'[^a-z\\s]', '', text)
    words = word_tokenize(text)
    processed_words = [ps.stem(word) for word in words if word not in stop_words]
    return " ".join(processed_words)

def analyze_document_similarity(files_list, similarity_threshold, similarity_method, top_n_pairs=3):
    if not files_list:
        return pd.DataFrame(), "Please upload at least one document file.", ""

    documents = []
    document_names = []

    for file_path in files_list:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            documents.append(content)
            document_names.append(os.path.basename(file_path))
        except UnicodeDecodeError:
            try:
                with open(file_path, 'r', encoding='latin-1') as f:
                    content = f.read()
                documents.append(content)
                document_names.append(os.path.basename(file_path))
            except Exception as e:
                return pd.DataFrame(), f"Error reading file {os.path.basename(file_path)} with both UTF-8 and latin-1 encodings: {e}", ""
        except Exception as e:
            return pd.DataFrame(), f"Error reading file {os.path.basename(file_path)}: {e}", ""

    if not documents:
        return pd.DataFrame(), "No readable content found in uploaded files.", ""

    if similarity_method == 'TF-IDF':
        preprocessed_docs = [preprocess_text_for_ui(doc) for doc in documents]
        tfidf_vectorizer = TfidfVectorizer()
        feature_matrix = tfidf_vectorizer.fit_transform(preprocessed_docs)
    elif similarity_method == 'Sentence-BERT':
        # Sentence-BERT works best with original text or lightly preprocessed text
        feature_matrix = sbert_model.encode(documents, show_progress_bar=False)
    else:
        return pd.DataFrame(), "Invalid similarity method selected.", ""

    cosine_sim_matrix = cosine_similarity(feature_matrix)

    cosine_sim_df = pd.DataFrame(
        cosine_sim_matrix,
        index=document_names,
        columns=document_names
    )

    similar_doc_pairs_info = []
    if len(document_names) > 1:
        for i in range(len(document_names)):
            for j in range(i + 1, len(document_names)):
                doc1_name = document_names[i]
                doc2_name = document_names[j]
                similarity = cosine_sim_matrix[i, j]
                similar_doc_pairs_info.append(((doc1_name, doc2_name), similarity))

        similar_doc_pairs_info.sort(key=lambda x: x[1], reverse=True)

    # --- Most Similar Pair Box HTML ---
    most_similar_pair_box_html = ""
    if similar_doc_pairs_info:
        (top_pair, top_sim) = similar_doc_pairs_info[0]
        box_background_color = ""
        box_message = ""
        if top_sim >= similarity_threshold:
            box_background_color = "#dc3545" # Red
            box_message = "Has More Similar Content"
        else:
            box_background_color = "#28a745" # Green
            box_message = "Within Bounds of Similarity"

        most_similar_pair_box_html = f"""
        <div style='
            background-color: {box_background_color};
            color: white;
            padding: 25px;
            border-radius: 10px;
            text-align: center;
            margin-bottom: 25px;
            box-shadow: 0 5px 15px rgba(0,0,0,0.3);
        '>
            <h3 style='margin-top: 0; margin-bottom: 12px; font-size: 1.6em;'>Most Similar Pair: {top_pair[0]} & {top_pair[1]}</h3>
            <p style='font-size: 3em; font-weight: bold; margin: 0;'>Similarity: {top_sim:.4f}</p>
            <p style='font-size: 1.4em; margin-top: 12px;'>{box_message}</p>
        </div>
        """
    else:
        most_similar_pair_box_html = "<p style='text-align: center; color: #f0f0f0;'>Upload at least two documents to find the most similar pair.</p>"

    top_similar_pairs_markdown = "### Top Similar Document Pairs (Document Level)\n\n"
    top_similar_pairs_markdown += f"**Similarity Threshold for Highlighting: {similarity_threshold:.2f}**\n\n"
    top_similar_pairs_markdown += f"**Similarity Method: {similarity_method}**\n\n"

    display_pairs = similar_doc_pairs_info[1:top_n_pairs+1]
    if not display_pairs:
        if not similar_doc_pairs_info:
            top_similar_pairs_markdown += "Upload at least two documents to see similar pairs."
        else:
            top_similar_pairs_markdown += "(Only one pair found, displayed above.)"
    else:
        for pair_idx, (pair, sim) in enumerate(display_pairs):

            color = '#FF0000' if sim >= similarity_threshold else '#008000' # Red for high, Green for safe
            font_size = '1.2em' if sim >= similarity_threshold else '1em'
            font_weight = 'bold' if sim >= similarity_threshold else 'normal'

            styled_sim = f"<span style='color: {color}; font-size: {font_size}; font-weight: {font_weight};'>{sim:.4f}</span>"

            top_similar_pairs_markdown += f"- **{pair[0]}** and **{pair[1]}**: Similarity: {styled_sim}\n"

    return cosine_sim_df, top_similar_pairs_markdown, most_similar_pair_box_html

In [ ]:
custom_css = """
html, body {
  height: 100%;
  width: 100%;
  margin: 0;
  padding: 0;
  overflow: hidden; /* Prevent scrollbars if content overflows */
  position: relative; /* Needed for pseudo-elements to position correctly */
}

body {
  background: linear-gradient(135deg, #2c3e50, #34495e, #2c3e50, #34495e); /* Darker animated background */
  background-size: 400% 400%;
  animation: gradientAnimation 15s ease infinite;
}

@keyframes gradientAnimation {
  0% { background-position: 0% 50%; }
  50% { background-position: 100% 50%; }
  100% { background-position: 0% 50%; }
}

/* Moving light elements */
body::before,
body::after {
  content: '';
  position: absolute;
  width: 30px; /* Increased size of the light elements */
  height: 30px; /* Increased size of the light elements */
  background: #f0f0f0; /* Light color */
  border-radius: 50%;
  opacity: 0.8; /* Increased opacity */
  filter: blur(5px); /* Adjusted blur for better visibility */
  animation: floatElements 25s infinite ease-in-out;
  z-index: -1; /* Behind the Gradio interface */
}

body::before {
  top: 10%;
  left: 5%;
  animation-delay: 0s;
  box-shadow: 200px 150px 0 0 rgba(240,240,240,0.3),
              -100px 300px 0 0 rgba(240,240,240,0.4),
              300px -50px 0 0 rgba(240,240,240,0.2),
              400px 200px 0 0 rgba(240,240,240,0.5);
}

body::after {
  top: 70%;
  left: 80%;
  animation-delay: 12s; /* Stagger animation start */
  box-shadow: -150px -100px 0 0 rgba(240,240,240,0.2),
              100px -200px 0 0 rgba(240,240,240,0.3),
              -250px 50px 0 0 rgba(240,240,240,0.4),
              -300px -250px 0 0 rgba(240,240,240,0.5);
}

@keyframes floatElements {
  0%   { transform: translate(0, 0) scale(1); opacity: 0.8; } /* Adjusted base opacity */
  25%  { transform: translate(50px, -70px) scale(1.1); opacity: 0.9; } /* Adjusted max opacity */
  50%  { transform: translate(100px, 0px) scale(1.2); opacity: 1.0; } /* Adjusted max opacity */
  75%  { transform: translate(50px, 70px) scale(1.1); opacity: 0.9; } /* Adjusted max opacity */
  100% { transform: translate(0, 0) scale(1); opacity: 0.8; } /* Adjusted base opacity */
}

.gradio-container {
  background: rgba(0, 0, 0, 0.7); /* Darker, semi-transparent background for content area */
  border-radius: 10px;
  box-shadow: 0px 4px 20px rgba(0, 0, 0, 0.5); /* Stronger shadow for dark theme */
  margin: 20px auto; /* Center the content block with margin */
  padding: 20px;
  max-width: 90%; /* Max width for better readability */
  box-sizing: border-box; /* Include padding in width calculation */
}

/* Ensure all text is light for a dark theme */
h1, h2, h3, h4, h5, h6, p, label, span { color: #f0f0f0 !important; }
.dark h1, .dark h2, .dark h3, .dark h4, .dark h5, .dark h6, .dark p, .dark label, .dark span { color: #f0f0f0 !important; }

button.primary:hover {
  background-color: #5d7a96; /* Lighter hover for dark theme */
  color: white;
}
"""

example_files_dir = "./gradio_examples/"
os.makedirs(example_files_dir, exist_ok=True)



interface = gr.Interface(
    fn=analyze_document_similarity,
    inputs=[
        gr.Files(label="Upload Documents (text files only)", file_count="multiple", type="filepath"),
        gr.Slider(minimum=0.0, maximum=1.0, value=0.7, step=0.01, label="Similarity Threshold for Highlighting"),
        gr.Radio(['TF-IDF', 'Sentence-BERT'], label="Choose Similarity Method", value='TF-IDF')
    ],
    outputs=[
        gr.DataFrame(label="Cosine Similarity Matrix (Document Level)"),
        gr.Markdown(label="Top Similar Document Pairs (Document Level)"),
        gr.Markdown(label="Most Similar Document Pair Analysis") # New output for the dedicated box
    ],
    title="Document Similarity Detector",
    description="Upload multiple text documents to analyze their textual similarity using TF-IDF or Sentence-BERT and Cosine Similarity at the document level. Results include the full document similarity matrix and top similar document pairs.",
    css=custom_css,
    theme='gradio/dark',
    examples=None
)

interface.launch(debug=True)